In [16]:
import torch
import torch.nn as nn
from transformers import AutoModelForTokenClassification

In [17]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [18]:
class QASpanDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModelForTokenClassification.from_pretrained('bert-base-uncased', num_labels=2)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids,
                             attention_mask=attention_mask)
        return outputs

In [19]:
model = QASpanDetector()
model = model.to(device)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForTokenClassification: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-u

In [20]:
import pickle

with open('data/debug_preprocessed2.pkl', 'rb') as f:
    train_encodings = pickle.load(f)

In [21]:
len(train_encodings['input_ids'])

753

In [22]:
len(train_encodings['input_ids'][0])

451

In [23]:
from preprocess import SquadDataset

train_dataset = SquadDataset(train_encodings)

In [24]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False)

In [25]:
import torch.nn.functional as F

def get_loss(reg_pred, obj_pred, reg_target, obj_target, answer_start):
    # Get answer length by get the the value at the answer start index of predict and target vector
    reg_pred = torch.gather(reg_pred, 1, answer_start).exp()
    reg_target = torch.gather(reg_target, 1, answer_start)

    reg_loss = F.mse_loss(reg_pred, reg_target)
    obj_loss = F.binary_cross_entropy_with_logits(obj_pred, obj_target)

    return reg_loss + obj_loss
    

In [26]:
MODEL_MAX_LENGTH = 512

In [27]:
import numpy as np

def create_labels(encodings):
        encodings['answer_length'] = np.array(encodings['end_positions'])\
         - np.array(encodings['start_positions']) + 1

        labels = np.zeros((len(encodings['input_ids']), MODEL_MAX_LENGTH, 
            2)) # num_example * seq_length * 2

        for example_idx, start in enumerate(encodings['start_positions']):
            if start < MODEL_MAX_LENGTH: # if the answer is not truncated
                labels[example_idx, start, 0] = 1
                labels[example_idx, start, 1] = encodings['answer_length'][example_idx]

        encodings['labels'] = labels

        return encodings

In [29]:
from torch.optim import AdamW

torch.manual_seed(0)
epochs = 3
print_every = 20
optim = AdamW(model.parameters(), lr=5e-5)
for epoch in range(epochs):
    # Set model in train mode
    model.train()
    loss_of_epoch = 0

    print("############Train############")
    for batch_idx, batch in enumerate(train_loader):
        batch = create_labels(batch)
        sentence_length = batch['input_ids'].size(1)
        batch_size = batch['input_ids'].size(0)

        answer_start = batch['start_positions'].reshape(batch_size, 1).to(device)        
        attention_mask = batch['attention_mask']
        attention_mask = F.pad(attention_mask, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)

        reg_target = batch['labels'][:, :, 1]
        obj_target = batch['labels'][:, :, 0]
        obj_target = torch.FloatTensor(obj_target).to(device)
        reg_target = torch.FloatTensor(reg_target).to(device)

        outputs = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))

        reg_predict = outputs['logits'][:, :, 1]
        obj_predict = outputs['logits'][:, :, 0]
        reg_predict = F.pad(reg_predict, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0).to(device)
        obj_predict = F.pad(obj_predict, (0, MODEL_MAX_LENGTH - sentence_length), 'constant', 0)
        obj_predict = obj_predict * attention_mask
        obj_predict = obj_predict.to(device)
        
        loss = get_loss(reg_predict, obj_predict, reg_target, obj_target, answer_start)
        print(loss)
        # loss.backward()
        # optim.step()
        loss_of_epoch += loss.item()
        if (batch_idx + 1) % print_every == 0:
            print("Batch {:} / {:}".format(batch_idx + 1, len(train_loader)))
            print("Loss:", round(loss.item(), 1), "\n")
        break
    break

############Train############
tensor(7.6652, grad_fn=<AddBackward0>)
